# 🫀 CardioIA — Fase 2: Diagnóstico Automatizado
## Parte 2: Classificador Básico de Texto para Triagem Clínica (TF-IDF + Scikit-Learn)

**Integrantes:**
- Giovani Saavedra (RM566797)
- Marcio Elifas (RM567871)

**Turma:** 2TIAOR — FIAP ON  
**Disciplina:** IA no Estetoscópio Digital  

### 1. Contexto e Objetivos
Nesta etapa, desenvolvemos um classificador inteligente de texto em linguagem natural para automatizar a triagem clínica de queixas cardiológicas.
O modelo analisa o relato do paciente e prediz o nível de risco:
- **Alto Risco:** Síndromes coronarianas agudas, arritmias graves ou insuficiência cardíaca descompensada.
- **Baixo Risco:** Sintomas musculoesqueléticos, fadiga tensional ou rotina ambulatorial.

In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

### 2. Carregamento e Exploração da Base de Dados
Carregamos o dataset simulado `dados/dataset_triagem.csv` contendo 50 relatos clínicos anotados e balanceados.

In [2]:
caminho_csv = os.path.join("dados", "dataset_triagem.csv")
df = pd.read_csv(caminho_csv)
print(f"Total de relatos no dataset: {len(df)}")
print("\nDistribuição das classes de risco:")
print(df["risco"].value_counts())
df.head(6)

Total de relatos no dataset: 50

Distribuição das classes de risco:
risco
alto risco     25
baixo risco    25


,relato,risco
0,sinto dor no peito e falta de ar intensa,alto risco
1,aperto forte no peito irradiando para o braco esquerdo,alto risco
2,falta de ar ao deitar e pernas muito inchadas,alto risco
3,coracao muito acelerado batimento descompassado e tontura,alto risco
4,queimacao retroesternal forte com suor frio,alto risco
5,dor no torax que piora muito ao caminhar ou subir escada,alto risco


### 3. Divisão Estratificada entre Treino e Teste
Separamos 75% dos dados para treinamento e 25% para teste, garantindo proporções idênticas com `stratify=y`.

In [3]:
X = df["relato"]
y = df["risco"]

X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
print(f"Amostras de treino: {len(X_treino)}")
print(f"Amostras de teste:  {len(X_teste)}")

Amostras de treino: 37
Amostras de teste:  13


### 4. Vetorização Textual com TF-IDF
Transformamos as frases em matrizes esparsas numéricas utilizando TF-IDF com unigramas e bigramas (`ngram_range=(1, 2)`).

In [4]:
vetorizador = TfidfVectorizer(ngram_range=(1, 2), lowercase=True)
X_treino_vec = vetorizador.fit_transform(X_treino)
X_teste_vec = vetorizador.transform(X_teste)

print(f"Dimensão da matriz de treino TF-IDF: {X_treino_vec.shape}")
print(f"Total de termos no vocabulário: {len(vetorizador.get_feature_names_out())}")
print(f"Exemplos de termos extraídos: {list(vetorizador.get_feature_names_out())[:12]}")

Dimensão da matriz de treino TF-IDF: (37, 391)
Total de termos no vocabulário: 391
Exemplos de termos extraídos: ['acelerado', 'acelerado batimento', 'acentuado', 'acentuado nos', 'acompanhada', 'acompanhada de', 'aguda', 'aguda no', 'alongamento', 'ao', 'ao apertar', 'ao caminhar']


### 5. Treinamento e Avaliação do Modelo 1: Regressão Logística

In [5]:
modelo_lr = LogisticRegression(random_state=42)
modelo_lr.fit(X_treino_vec, y_treino)
pred_lr = modelo_lr.predict(X_teste_vec)
acc_lr = accuracy_score(y_teste, pred_lr)

print(f"Acurácia Global no Teste: {acc_lr * 100:.2f}%\n")
print("Relatório de Classificação:")
print(classification_report(y_teste, pred_lr))
print("Matriz de Confusão:")
print(confusion_matrix(y_teste, pred_lr))

Acurácia Global no Teste: 92.31%

Relatório de Classificação:
              precision    recall  f1-score   support

  alto risco       1.00      0.86      0.92         7
 baixo risco       0.86      1.00      0.92         6

    accuracy                           0.92        13
   macro avg       0.93      0.93      0.92        13
weighted avg       0.93      0.92      0.92        13

Matriz de Confusão:
[[6 1]
 [0 6]]


### 6. Treinamento e Avaliação do Modelo 2: Árvore de Decisão

In [6]:
modelo_dt = DecisionTreeClassifier(random_state=42, max_depth=5)
modelo_dt.fit(X_treino_vec, y_treino)
pred_dt = modelo_dt.predict(X_teste_vec)
acc_dt = accuracy_score(y_teste, pred_dt)

print(f"Acurácia Global no Teste: {acc_dt * 100:.2f}%\n")
print("Relatório de Classificação:")
print(classification_report(y_teste, pred_dt))
print("Matriz de Confusão:")
print(confusion_matrix(y_teste, pred_dt))

Acurácia Global no Teste: 92.31%

Relatório de Classificação:
              precision    recall  f1-score   support

  alto risco       0.88      1.00      0.93         7
 baixo risco       1.00      0.83      0.91         6

    accuracy                           0.92        13
   macro avg       0.94      0.92      0.92        13
weighted avg       0.93      0.92      0.92        13

Matriz de Confusão:
[[7 0]
 [1 5]]


### 7. Discussão Crítica de Desempenho e Viés em Triagem Médica

> **Síntese Comparativa:** Ambos os modelos empatam rigorosamente com **92,31% de acurácia global (12 acertos em 13)** no conjunto de teste independente, mas apresentam perfis de erro clínicos complementares. A **Árvore de Decisão** priorizou a sensibilidade, atingindo **Recall de 100% para Alto Risco** ao custo de 1 Falso Positivo (encaminhou 1 caso leve ao PS, o que é clinicamente seguro em emergência). Já a **Regressão Logística** obteve **Precisão de 100% para Alto Risco**, com 1 Falso Negativo. Na prática hospitalar, a Regressão Logística é a preferida para produção porque fornece **probabilidades contínuas e calibradas (`predict_proba`)**, permitindo à equipe médica ajustar o limiar de decisão (*threshold*) para elevar a sensibilidade a 100%, enquanto a Árvore de Decisão opera por cortes discretos rígidos que tendem a sobreajustar (*overfitting*) em vocabulários mais amplos.

### 8. Teste Prático em Frases Inéditas (Predição em Tempo Real com Probabilidades)

In [7]:
frases_novas = [
    "Estou sentindo uma dor muito forte no meio do peito e falta de ar",
    "Tive apenas uma leve pontada muscular nas costas apos a academia",
    "Aperto insuportavel no torax com suor frio e sensacao de desmaio",
    "Desconforto nos ombros por ficar muito tempo sentado trabalhando"
]

frases_vec = vetorizador.transform(frases_novas)
predicoes = modelo_lr.predict(frases_vec)
probabilidades = modelo_lr.predict_proba(frases_vec)

print("=== RESULTADOS DA PREDIÇÃO EM TEMPO REAL ===")
for frase, pred, prob in zip(frases_novas, predicoes, probabilidades):
    conf = prob.max() * 100
    print(f"\nFrase: \"{frase}\"")
    print(f"--> Classificação: {pred.upper()} (Confiança: {conf:.1f}%)")

=== RESULTADOS DA PREDIÇÃO EM TEMPO REAL ===

Frase: "Estou sentindo uma dor muito forte no meio do peito e falta de ar"
--> Classificação: ALTO RISCO (Confiança: 67.5%)

Frase: "Tive apenas uma leve pontada muscular nas costas apos a academia"
--> Classificação: BAIXO RISCO (Confiança: 70.4%)

Frase: "Aperto insuportavel no torax com suor frio e sensacao de desmaio"
--> Classificação: ALTO RISCO (Confiança: 60.7%)

Frase: "Desconforto nos ombros por ficar muito tempo sentado trabalhando"
--> Classificação: BAIXO RISCO (Confiança: 58.5%)
